In [ ]:
# Imports
import os
import re
import time
import requests
import math
import numpy as np
import rasterio
from rasterio.merge import merge
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterio.mask import mask
from rasterio.windows import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
import h5py
from datetime import datetime
from requests.auth import HTTPBasicAuth
from rasterio.errors import WindowError

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Setup do ambiente e da authenticação com a API

In [ ]:
# Pasta no Google Drive (resultado final)
drive_output_dir = "/content/drive/MyDrive/viirs_output"
os.makedirs(drive_output_dir, exist_ok=True)

# Pasta temporária local (mais rápido)
temp_dir = "/content/temp_viirs"
os.makedirs(temp_dir, exist_ok=True)

print("Drive:", drive_output_dir)
print("Temp:", temp_dir)

DATA_DIR = temp_dir
OUT_DIR = drive_output_dir

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Usuario e senha da NASA.
auth = HTTPBasicAuth("USER", "SENHA")
DATE = "2025-04-10"

# AOI
lon_min, lon_max = -53.5, -45
lat_min, lat_max = -19.5, -11.8

# Folder do shapefile no drive
SHAPEFILE = "/content/drive/MyDrive/GO/GO_UF_2024.shp"


Drive: /content/drive/MyDrive/viirs_output
Temp: /content/temp_viirs


In [ ]:

def get_doy(date):
    return datetime.strptime(date, "%Y-%m-%d").strftime("%j")


In [ ]:
class SessionWithHeaderRedirection(requests.Session):
    AUTH_HOST = 'urs.earthdata.nasa.gov'

    def __init__(self, auth):
        super().__init__()
        self.auth = auth

    def rebuild_auth(self, prepared_request, response):
        """
        Mantém auth mesmo após redirect entre hosts
        """
        headers = prepared_request.headers
        url = prepared_request.url

        if 'Authorization' in headers:
            original_host = requests.utils.urlparse(response.request.url).hostname
            redirect_host = requests.utils.urlparse(url).hostname

            if (original_host != redirect_host) and (redirect_host != self.AUTH_HOST):
                del headers['Authorization']

In [ ]:
# Usa sessão para poder baixar dados mesmo tendo autenticação
def get_session():

    session = SessionWithHeaderRedirection(auth)
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    return session

## Download com seleção automática

In [ ]:
def safe_download(session, url, out, max_retries=3):

    for attempt in range(max_retries):

        try:
            print(f"Tentativa {attempt+1}:", url.split("/")[-1])

            r = session.get(url, allow_redirects=True, timeout=60)

            if r.status_code != 200:
                raise Exception(f"HTTP {r.status_code}")

            content_type = r.headers.get("Content-Type", "")

            if "text/html" in content_type:
                raise Exception("Recebeu HTML (auth falhou no redirect)")

            with open(out, "wb") as f:
                f.write(r.content)

            size = os.path.getsize(out)

            return out

        except Exception as e:

            print("Erro:", e)

            # remove arquivo corrompido
            if os.path.exists(out):
                os.remove(out)

            if attempt < max_retries - 1:
                print("Retrying...\n")
                time.sleep(2)
            else:
                raise Exception(f"Falha após {max_retries} tentativas: {url}")

In [ ]:
def download_viirs(product):

    session = get_session()

    year = DATE[:4]
    doy = get_doy(DATE)

    base_url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/{product}/{year}/{doy}"
    json_url = base_url + ".json"

    print("Acessando:", json_url)

    r = session.get(json_url)

    if r.status_code != 200:
        raise Exception(f"Erro HTTP {r.status_code}")

    data = r.json()
    files = data.get("content", [])

    if not files:
        raise Exception(f"Nenhum dado disponível para {DATE}")

    # apenas HDF5
    hdfs = [f for f in files if f["name"].endswith(".h5")]

    print(f"{len(hdfs)} tiles encontrados")

    # filtro espacial
    hdfs = filter_tiles_by_bbox(hdfs, lon_min, lon_max, lat_min, lat_max)

    if not hdfs:
        print("Nenhum tile no bbox → usando todos")
        hdfs = [f for f in files if f["name"].endswith(".h5")]

    print(f"{len(hdfs)} tiles após filtro")

    paths = []

    for f in hdfs:

        name = f["name"]
        file_url = f["downloadsLink"]

        out = os.path.join(DATA_DIR, name)

        # valida arquivo existente
        def is_valid(file):
            return os.path.exists(file) and os.path.getsize(file) > 1_000_000

        if not is_valid(out):

            print("Downloading:", name)

            safe_download(session, file_url, out)

        else:
            print("Usando cache:", name)

        paths.append(out)

    return paths

In [ ]:
def filter_tiles_by_bbox(files, lon_min, lon_max, lat_min, lat_max):

    valid = []

    for f in files:

        # handle both cases
        if isinstance(f, dict):
            name = f["name"]
        else:
            name = os.path.basename(f)

        match = re.search(r"h(\d{2})v(\d{2})", name)

        if not match:
            continue

        h = int(match.group(1))
        v = int(match.group(2))

        # tile bounds (approx)
        tile_lon_min = -180 + h * 10
        tile_lon_max = tile_lon_min + 10

        tile_lat_max = 90 - v * 10
        tile_lat_min = tile_lat_max - 10

        intersects = not (
            tile_lon_max < lon_min or
            tile_lon_min > lon_max or
            tile_lat_max < lat_min or
            tile_lat_min > lat_max
        )

        if intersects:
            valid.append(f)

    print(f"{len(valid)} tiles após filtro")

    return valid

## Buscar banda automaticamente

In [ ]:

def find_dataset(hdf, band):

    with rasterio.open(hdf) as src:

        subs = src.subdatasets

        # debug opcional
        # print("\n".join(subs))

        mapping = {
            "M5": "SurfReflect_M5",
            "M11": "SurfReflect_M11",
            "Emis_14": "Emis_14",
            "Emis_15": "Emis_15",
            "LST_1KM": "LST_1KM"
        }

        target = mapping.get(band, band)

        matches = [s for s in subs if target in s]

        if not matches:
            raise Exception(f"Banda '{band}' não encontrada no HDF")

        return matches[0]


## Extração + reprojeção real

In [ ]:
def extract_reproject(subdataset_path, out_tif):

    with rasterio.open(subdataset_path) as src:

        if src.crs is None:
            raise Exception("Raster sem CRS (subdataset inválido)")

        transform, width, height = calculate_default_transform(
            src.crs, "EPSG:4326", src.width, src.height, *src.bounds
        )

        kwargs = {
            "driver": "GTiff",
            "height": height,
            "width": width,
            "count": 1,
            "dtype": "float32",
            "crs": "EPSG:4326",
            "transform": transform,
            "nodata": np.nan
        }

        with rasterio.open(out_tif, "w", **kwargs) as dst:

            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs="EPSG:4326",
                resampling=Resampling.bilinear
            )
    with rasterio.open(out_tif) as test:
      print(test.crs)
      print(test.bounds)

    return out_tif

In [ ]:
def process_tiles(files, band_name, prefix):

    outputs = []

    for i, f in enumerate(files):

        print(f"Processando {band_name} | tile {i+1}/{len(files)}")

        out_path = os.path.join(DATA_DIR, f"{prefix}_{i}.tif")

        subdataset = find_dataset(f, band_name)

        tif = extract_reproject(subdataset, out_path)
        tif = crop(tif)

        # Ignora Tiles fora da AOI
        if tif is None:
            continue

        outputs.append(tif)

    return outputs

In [ ]:
def mosaic(tifs, out_path):

    srcs = [rasterio.open(t) for t in tifs]

    mosaic_arr, transform = merge(
        srcs,
        resampling=Resampling.bilinear  # importante
    )

    meta = srcs[0].meta.copy()
    meta.update({
        "height": mosaic_arr.shape[1],
        "width": mosaic_arr.shape[2],
        "transform": transform
    })

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic_arr)

    for s in srcs:
        s.close()

    return out_path

## Crop

In [ ]:
def crop(tif):

    gdf = gpd.read_file(SHAPEFILE)

    with rasterio.open(tif) as src:

        gdf = gdf.to_crs(src.crs)
        geom = gdf.geometry.values

        print("Raster CRS:", src.crs)
        print("Shape CRS:", gdf.crs)

        try:
            out, transform = mask(src, geom, crop=True)

        except ValueError:
            # NÃO INTERSECTA → IGNORA TILE
            print(f" Tile ignorado (sem interseção): {os.path.basename(tif)}")
            return None

        meta = src.meta.copy()
        meta.update({
            "height": out.shape[1],
            "width": out.shape[2],
            "transform": transform
        })

    cropped = tif.replace(".tif", "_clip.tif")

    with rasterio.open(cropped, "w", **meta) as dst:
        dst.write(out)

    return cropped

In [ ]:

def read(tif):
    with rasterio.open(tif) as src:
        return src.read(1)


In [ ]:
def preview_geo(tif, title, label, cmap="turbo", vmin=None, vmax=None):

    with rasterio.open(tif) as src:

        arr = src.read(1)
        bounds = src.bounds

    if vmin is None or vmax is None:
        vmin = np.nanpercentile(arr, 2)
        vmax = np.nanpercentile(arr, 98)

    plt.figure(figsize=(7,6))

    img = plt.imshow(
        arr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        interpolation="bilinear"
    )

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")

    cbar = plt.colorbar(img)
    cbar.set_label(label)

    plt.title(title)

    path = os.path.join(OUT_DIR, f"{title}.png")
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

## Pipeline completo

In [ ]:

# DOWNLOAD
sr_files = download_viirs("VNP09GA")
th_files = download_viirs("VNP21A1D")

# PROCESSAMENTO POR TILE POR BANDA
m5_tiles = process_tiles(sr_files, "M5", "m5")
m11_tiles = process_tiles(sr_files, "M11", "m11")

m13_tiles = process_tiles(th_files, "Emis_14", "m13")
m15_tiles = process_tiles(th_files, "LST_1KM", "m15")
m16_tiles = process_tiles(th_files, "Emis_15", "m16")

# MOSAICO FINAL
m5_path = mosaic(m5_tiles, os.path.join(DATA_DIR, "m5_mosaic.tif"))
m11_path = mosaic(m11_tiles, os.path.join(DATA_DIR, "m11_mosaic.tif"))

m13_path = mosaic(m13_tiles, os.path.join(DATA_DIR, "m13_mosaic.tif"))
m15_path = mosaic(m15_tiles, os.path.join(DATA_DIR, "m15_mosaic.tif"))
m16_path = mosaic(m16_tiles, os.path.join(DATA_DIR, "m16_mosaic.tif"))

In [ ]:

# Cálculo dos indices
m5 = read(m5_path) / 10000
m11 = read(m11_path) / 10000
m13 = read(m13_path)
m15 = read(m15_path)
m16 = read(m16_path)

ndi = (m5 - m11) / (m5 + m11 + 1e-6)
anom = np.clip(m13 - m15, -10, 50)
lst = m15 + 0.5*(m15 - m16)

# Faz o preview.png de cada indice, usa Min e Máx definido manualmente
preview_geo(ndi, "NDI", "Índice", cmap="RdYlBu_r", vmin=-1, vmax=1)
preview_geo(anom, "Anomalia", "K", cmap="RdYlBu_r", vmin=-10, vmax=50)
preview_geo(lst, "LST", "°C", cmap="Oranges", vmin=0, vmax=50)

print("Finalizado:", OUT_DIR)


## Pipeline para zoom em coordenada específica, integração ao Modelo

In [ ]:
# Criação da BBOX usando a coordenada passada como centro, expande N-km para cada lado
def point_to_bbox(lat, lon, km=2):

    # approx conversion
    dlat = km / 111.0
    dlon = km / (111.0 * math.cos(math.radians(lat)))

    return (
        lon - dlon,
        lon + dlon,
        lat - dlat,
        lat + dlat
    )

In [ ]:
def read_array(tif):

    with rasterio.open(tif) as src:
        arr = src.read(1).astype("float32")

        # handle nodata properly
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

    return arr

In [ ]:
def crop_bbox(tif, lon_min, lon_max, lat_min, lat_max):

    with rasterio.open(tif) as src:

        try:
            window = from_bounds(
                lon_min, lat_min, lon_max, lat_max,
                src.transform
            )

            data = src.read(1, window=window)

        except WindowError:
            print(" bbox fora do tile:", tif)
            return None

        transform = src.window_transform(window)

        meta = src.meta.copy()
        meta.update({
            "height": data.shape[0],
            "width": data.shape[1],
            "transform": transform
        })

    out = tif.replace(".tif", "_crop.tif")

    with rasterio.open(out, "w", **meta) as dst:
        dst.write(data, 1)

    return out

In [ ]:
def run_point(lat, lon):

    print(f"Processing point: {lat}, {lon}")

    lon_min, lon_max, lat_min, lat_max = point_to_bbox(lat, lon, km=2)

    # download products
    sr_files = download_viirs("VNP09GA")
    th_files = download_viirs("VNP21A1D")

    # filter tiles dynamically
    sr_files = filter_tiles_by_bbox(sr_files, lon_min, lon_max, lat_min, lat_max)
    th_files = filter_tiles_by_bbox(th_files, lon_min, lon_max, lat_min, lat_max)

    print("Tiles used:", len(sr_files))

    # process bands
    m5_tiles = process_tiles(sr_files, "M5", "m5")
    m11_tiles = process_tiles(sr_files, "M11", "m11")

    m15_tiles = process_tiles(th_files, "LST_1KM", "lst")
    m16_tiles = process_tiles(th_files, "Emis_15", "e15")

    # crop to point bbox
    m5 = [crop_bbox(t, lon_min, lon_max, lat_min, lat_max) for t in m5_tiles]
    m11 = [crop_bbox(t, lon_min, lon_max, lat_min, lat_max) for t in m11_tiles]

    m15 = [crop_bbox(t, lon_min, lon_max, lat_min, lat_max) for t in m15_tiles]
    m16 = [crop_bbox(t, lon_min, lon_max, lat_min, lat_max) for t in m16_tiles]

    # read (use first tile — small area)
    m5 = read_array(m5[0]) / 10000.0
    m11 = read_array(m11[0]) / 10000.0

    m15 = read_array(m15[0])
    m16 = read_array(m16[0])

    # indices
    ndi = (m5 - m11) / (m5 + m11 + 1e-6)
    lst = m15 + 0.5 * (m15 - m16)

    # save temporary tif for plotting
    ndi_tif = os.path.join(DATA_DIR, "ndi_tmp.tif")
    lst_tif = os.path.join(DATA_DIR, "lst_tmp.tif")

    # reuse georef from m5 tile
    with rasterio.open(m5_tiles[0]) as ref:

        meta = ref.meta.copy()
        meta.update({"dtype": "float32", "count": 1})

        with rasterio.open(ndi_tif, "w", **meta) as dst:
            dst.write(ndi.astype("float32"), 1)

        with rasterio.open(lst_tif, "w", **meta) as dst:
            dst.write(lst.astype("float32"), 1)

    # previews with coords
    preview_geo(ndi_tif, "NDI_point", "Index", cmap="RdYlBu_r", vmin=-1, vmax=1)
    preview_geo(lst_tif, "LST_point", "°C", cmap="Oranges", vmin=0, vmax=50)

    print("Done.")

In [ ]:
# Usar a função e zoom nas coordenadas: Lat, Lon: -16.68, -49.25

run_point(-16.68, -49.25)

Processing point: -16.68, -49.25
Acessando: https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP09GA/2025/100.json
460 tiles encontrados
2 tiles após filtro
2 tiles após filtro
Usando cache: VNP09GA.A2025100.h12v10.002.2025101100643.h5
Usando cache: VNP09GA.A2025100.h13v10.002.2025101100336.h5
Acessando: https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP21A1D/2025/100.json
359 tiles encontrados
2 tiles após filtro
2 tiles após filtro
Usando cache: VNP21A1D.A2025100.h12v10.002.2025101100009.h5
Usando cache: VNP21A1D.A2025100.h13v10.002.2025101095748.h5
1 tiles após filtro
1 tiles após filtro
Tiles used: 1
Processando M5 | tile 1/1


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


EPSG:4326
BoundingBox(left=-53.20888861840601, bottom=-19.999943112486665, right=-40.617496832349175, top=-9.999999999104968)
Raster CRS: EPSG:4326
Shape CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Processando M11 | tile 1/1


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


EPSG:4326
BoundingBox(left=-53.20888861840601, bottom=-19.999943112486665, right=-40.617496832349175, top=-9.999999999104968)
Raster CRS: EPSG:4326
Shape CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Processando LST_1KM | tile 1/1


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


EPSG:4326
BoundingBox(left=-53.20888861840601, bottom=-19.999943112486665, right=-40.617496832349175, top=-9.999999999104968)
Raster CRS: EPSG:4326
Shape CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Processando Emis_15 | tile 1/1


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


EPSG:4326
BoundingBox(left=-53.20888861840601, bottom=-19.999943112486665, right=-40.617496832349175, top=-9.999999999104968)
Raster CRS: EPSG:4326
Shape CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Done.
